<a href="https://colab.research.google.com/github/YuriArduino/Estudos_Artificial_Intelligence/blob/Lang_chain/Lang_chain_atualizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ## Roteiro de Estudos Otimizado: LangChain com Google Gemini

**Criado por:** Yuri Arduino

**Objetivo:** Este notebook é um guia atualizado e otimizado para estudos de LangChain, focado na construção de sistemas de RAG (Retrieval-Augmented Generation).

**Otimizações aplicadas:**
- **Organização:** Estrutura modular com explicações claras.
- **Pydantic v2:** Uso de `pydantic-settings` para gerenciamento de configurações.
- **Performance:** Otimização para GPU em modelos de embedding locais.
- **Boas Práticas:** Código limpo, comentado e seguindo as versões mais recentes das bibliotecas.

---

#1. Instalação de Dependências

 Instalação das bibliotecas essenciais:
 - langchain e langchain-google-genai: Para orquestração e integração com a API do Gemini.
 - pypdf e unstructured: Para carregar e extrair texto de documentos PDF.
 - faiss-cpu e chromadb: Vector stores para armazenar e buscar embeddings localmente.
 - sentence-transformers: Necessário para os modelos de embedding do Hugging Face.

In [1]:
# Célula de Instalação Completa
# Garante que todos os pacotes necessários, incluindo os da comunidade, sejam instalados.
!pip install -q --upgrade langchain langchain-core langchain-community langchain-google-genai pypdf sentence-transformers faiss-cpu chromadb "pydantic-settings>2.0.0"

print("✅ Bibliotecas instaladas com sucesso!")

✅ Bibliotecas instaladas com sucesso!


In [2]:
import langchain
print(langchain.__version__)

1.0.5


##1.1. Configuração do Ambiente (Logging e API Keys) (Código)

In [3]:
import os
import logging
from datetime import datetime
import pytz
from google.colab import userdata

# --- Configuração do Logging com Fuso Horário de Brasília ---
# Um bom sistema de logs é essencial para depurar e entender o que está acontecendo.
brasilia_tz = pytz.timezone("America/Sao_Paulo")

class TZFormatter(logging.Formatter):
    def formatTime(self, record, datefmt=None):
        dt = datetime.fromtimestamp(record.created, tz=brasilia_tz)
        return dt.strftime(datefmt or "%H:%M:%S")

handler = logging.StreamHandler()
handler.setFormatter(TZFormatter("%(asctime)s | %(levelname)-7s | %(message)s"))
logging.basicConfig(level=logging.INFO, handlers=[handler], force=True)
logger = logging.getLogger(__name__)

logger.info("Logging configurado com sucesso - Horário de Brasília (UTC-3).")

# --- Configuração da API Key do Google ---
# Carrega a chave da API do Gemini a partir dos "Secrets" do Google Colab.
# Esta é uma prática de segurança para não expor suas chaves no código.
try:
    api_key = userdata.get("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = api_key
    logger.info("GEMINI_API_KEY carregada com sucesso.")
except Exception as e:
    logger.error("Chave GEMINI_API_KEY não encontrada nos Secrets do Colab. Por favor, adicione-a.")
    raise e

12:52:54 | INFO    | Logging configurado com sucesso - Horário de Brasília (UTC-3).
12:52:54 | INFO    | GEMINI_API_KEY carregada com sucesso.


##1.2. Carregador do Modelo LLM

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

def carregar_llm(model: str = "gemini-2.5-flash", temperature: float = 0):
    """
    Carrega e configura o modelo LLM do Google Gemini.

    Args:
        model (str): O nome do modelo a ser usado.
        temperature (float): O nível de criatividade do modelo (0 = determinístico).

    Returns:
        ChatGoogleGenerativeAI: Instância do modelo LLM pronto para uso.
    """
    try:
        logger.info(f"--- Carregando modelo LLM: {model} ---")
        llm = ChatGoogleGenerativeAI(model=model, temperature=temperature)
        logger.info(f"✅ Modelo '{model}' carregado com sucesso!")
        return llm
    except Exception as e:
        logger.error(f"❌ Erro ao carregar o modelo LLM: {e}")
        return None

# Carrega o modelo principal que será usado no notebook
llm_model = carregar_llm()

12:53:15 | INFO    | --- Carregando modelo LLM: gemini-2.5-flash ---
12:53:15 | INFO    | ✅ Modelo 'gemini-2.5-flash' carregado com sucesso!


# Seção 1: Fundamentos de RAG (Retrieval-Augmented Generation)

Nesta seção, vamos explorar a diferença entre uma abordagem de LLM tradicional e uma abordagem RAG.

**Cenário:** Queremos perguntar ao nosso LLM sobre a política de home office de uma empresa fictícia. O modelo, por padrão, não tem essa informação.

**1. Abordagem Tradicional (Sem RAG):** O LLM tentará responder com base em seu conhecimento geral, o que geralmente resulta em uma resposta genérica ou incorreta.

**2. Abordagem RAG:** Nós fornecemos ao LLM o documento exato com a política (o "contexto") e pedimos que ele baseie sua resposta *apenas* nesse contexto.

##Exemplo Prático - Abordagem Tradicional

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Pergunta sobre um conhecimento específico que o LLM não possui
pergunta = "Qual é a política de home office da nossa empresa?"

# 1. Prompting Tradicional
prompt_tradicional = ChatPromptTemplate.from_template(
    "Responda a seguinte pergunta: {pergunta}"
)

# 2. Criação da Cadeia (Chain) com LangChain Expression Language (LCEL)
# A sintaxe com "|" é a forma moderna e recomendada de criar cadeias.
chain_tradicional = prompt_tradicional | llm_model

# 3. Execução da Cadeia
logger.info("Executando a cadeia tradicional (sem RAG)...")
resposta_tradicional = chain_tradicional.invoke({"pergunta": pergunta})

print("\n" + "="*50)
print(f"Pergunta: {pergunta}")
print(f"Resposta (Sem RAG): {resposta_tradicional.content}")
print("="*50)

logger.warning("Observe que a resposta é genérica, pois o modelo não conhece a política específica.")

12:53:15 | INFO    | Executando a cadeia tradicional (sem RAG)...
12:53:22 | WARNING | Observe que a resposta é genérica, pois o modelo não conhece a política específica.



Pergunta: Qual é a política de home office da nossa empresa?
Resposta (Sem RAG): Como uma inteligência artificial, eu não tenho acesso às políticas internas específicas da sua empresa.

Para saber qual é a política de home office da **sua empresa**, você deve consultar as seguintes fontes:

1.  **Departamento de Recursos Humanos (RH):** É a fonte mais confiável para obter informações sobre políticas da empresa. Eles podem fornecer o documento oficial ou explicar os detalhes.
2.  **Seu Gestor Direto/Supervisor:** Ele(a) deve estar ciente da política e pode orientá-lo sobre como ela se aplica à sua equipe e função.
3.  **Portal Interno/Intranet da Empresa:** Muitas empresas têm um portal online onde todas as políticas, manuais do funcionário e comunicados são publicados. Procure por seções como "Políticas", "RH", "Trabalho Remoto" ou "Home Office".
4.  **Manual do Funcionário/Guia de Políticas:** Se a empresa possui um manual físico ou digital, a política de home office provavelmente es

##Exemplo Prático - Abordagem RAG

In [6]:
import time
from pathlib import Path
from typing import List
from urllib.parse import unquote
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# --- 1. Lógica de Download (Robusta) ---
class DocumentDownloader:
    def __init__(self, persist_dir: str = "/content/data"):
        self.persist_dir = Path(persist_dir)
        self.persist_dir.mkdir(parents=True, exist_ok=True)

    def _get_filename_from_url(self, url: str) -> str:
        return unquote(url.split("/")[-1].split("?")[0])

    def _convert_github_url(self, url: str) -> str:
        if "github.com" in url and "/blob/" in url:
            logger.info("URL do GitHub detectada. Convertendo para formato raw...")
            return url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
        return url

    def download(self, url: str) -> Path:
        raw_url = self._convert_github_url(url)
        filename = self._get_filename_from_url(raw_url)
        filepath = self.persist_dir / filename

        if filepath.exists():
            logger.info(f"Arquivo '{filename}' já existe. Usando cache.")
            return filepath

        try:
            logger.info(f"Baixando '{filename}' de {raw_url}...")
            response = requests.get(raw_url, timeout=30)
            response.raise_for_status()
            filepath.write_bytes(response.content)
            logger.info(f"✅ Arquivo salvo em: {filepath}")
            return filepath
        except requests.RequestException as e:
            logger.error(f"❌ Falha no download de {raw_url}: {e}")
            return None

# --- 2. Execução do Pipeline RAG ---

# LISTA DE URLs CORRIGIDA (adicionando a branch 'Lang_chain')
urls = [
    "https://github.com/YuriArduino/Estudos_Artificial_Intelligence/blob/Dados/politica_home_office.pdf",
    "https://github.com/YuriArduino/Estudos_Artificial_Intelligence/blob/Dados/relatorio_vendas.pdf"
]

# Vamos usar apenas o primeiro PDF para este exemplo
pdf_url = urls[0]

downloader = DocumentDownloader()
pdf_path = downloader.download(pdf_url)

if pdf_path:
    logger.info(f"Carregando texto do PDF: {pdf_path}...")
    loader = PyPDFLoader(str(pdf_path))
    documento_pdf = loader.load()
    contexto_empresa = documento_pdf[0].page_content
    logger.info("✅ Contexto extraído do PDF com sucesso.")

    pergunta = "Qual é a política de home office da nossa empresa?"

    prompt_rag = ChatPromptTemplate.from_template(
        """
        Você é um assistente de RH. Responda a pergunta do usuário baseando-se estritamente no contexto fornecido.
        Se a informação não estiver no contexto, diga "Com base no documento, não tenho informações sobre isso."

        **Contexto:**
        {contexto}

        **Pergunta:**
        {pergunta}
        """
    )

    # O `llm_model` deve ter sido criado em uma célula anterior
    chain_rag = (
        {"contexto": lambda x: contexto_empresa, "pergunta": lambda x: x["pergunta"]}
        | prompt_rag
        | llm_model
    )

    logger.info("Invocando a cadeia RAG...")
    resposta_rag = chain_rag.invoke({"pergunta": pergunta})

    print("\n" + "="*50)
    print(f"Pergunta: {pergunta}")
    print(f"Resposta (Com RAG): {resposta_rag.content}")
    print("="*50)
else:
    logger.error("Pipeline RAG não pôde ser executado pois o download do PDF falhou.")

12:53:25 | INFO    | NumExpr defaulting to 2 threads.
12:53:39 | INFO    | TensorFlow version 2.19.0 available.
12:53:39 | INFO    | JAX version 0.7.2 available.
12:53:40 | INFO    | URL do GitHub detectada. Convertendo para formato raw...
12:53:40 | INFO    | Arquivo 'politica_home_office.pdf' já existe. Usando cache.
12:53:40 | INFO    | Carregando texto do PDF: /content/data/politica_home_office.pdf...
12:53:40 | INFO    | ✅ Contexto extraído do PDF com sucesso.
12:53:40 | INFO    | Invocando a cadeia RAG...



Pergunta: Qual é a política de home office da nossa empresa?
Resposta (Com RAG): A política de trabalho remoto e híbrido da Empresa XYZ estabelece as seguintes diretrizes:

*   **Modalidade Padrão:** É híbrida, compreendendo 3 (três) dias de trabalho remoto (home office) e 2 (dois) dias de trabalho presencial no escritório, por semana.
*   **Dias Presenciais:** Serão definidos em comum acordo entre a equipe e o gestor, priorizando as terças-feiras para reuniões de alinhamento geral da equipe. A presença no escritório neste dia é obrigatória.
*   **Elegibilidade:** Todos os funcionários em tempo integral, que completaram o período de experiência de 90 dias e cujas funções são compatíveis com o trabalho remoto, são elegíveis. A aprovação final está sujeita ao acordo com o gestor direto.
*   **Horário Flexível:** A jornada de trabalho de 8 horas diárias pode ser cumprida com flexibilidade, iniciando entre 07:00 e 10:00. O horário de 'core time', no qual todos devem estar disponíveis onli

# Seção 2: Embeddings e Vector Stores - O Cérebro da Memória do RAG

Para que um sistema RAG funcione com dezenas ou milhares de documentos, é impossível enviar todos eles como contexto para o LLM a cada pergunta. Precisamos de uma estratégia inteligente para encontrar, em milissegundos, os trechos de informação mais relevantes para a dúvida do usuário.

É aqui que a mágica acontece, através de dois componentes fundamentais: **Embeddings** e **Vector Stores**.

### O que são Embeddings? A Tradução da Semântica

Um embedding é a representação numérica do significado de um texto. Um modelo de embedding especializado lê um trecho de texto e o transforma em um **vetor** (uma longa lista de números).

A principal característica é que textos com significados parecidos, como *"qual a regra para tirar férias?"* e *"como solicito meus dias de descanso?"*, terão vetores matematicamente próximos no espaço vetorial. É como dar um endereço (coordenadas) para o significado de cada pedaço de texto.

### O que são Vector Stores? A Biblioteca Otimizada

Um Vector Store é um tipo de banco de dados construído especificamente para armazenar esses vetores e realizar buscas por similaridade em altíssima velocidade. Quando uma nova pergunta chega, nós a transformamos em um vetor e pedimos ao Vector Store para "encontrar os vetores mais próximos", que correspondem aos documentos mais relevantes para responder àquela pergunta.


### Nossas Ferramentas de Trabalho

Nesta seção, vamos colocar a mão na massa e comparar três das mais populares e versáteis soluções de Vector Stores, cada uma adequada para um cenário diferente:

1.  **FAISS:** Desenvolvido pelo Facebook AI, é extremamente rápido e opera **em memória**. Perfeito para prototipagem rápida e cenários onde os dados não são massivos.
2.  **ChromaDB:** Uma solução moderna e fácil de usar que **persiste os dados em disco**, facilitando o reuso sem a necessidade de reprocessar tudo. Oferece um poderoso sistema de filtragem por metadados.
3.  **Pinecone:** Um serviço gerenciado **na nuvem**, projetado para produção e grande escala. Oferece alta disponibilidade, escalabilidade e recursos avançados para aplicações robustas.

###Instalações e Preparação

In [7]:
# Instala as bibliotecas específicas para esta seção:
# - faiss-cpu: Para o vector store FAISS (versão para CPU)
# - chromadb: Para o vector store Chroma
# - langchain-pinecone & pinecone-client: Para a integração com o serviço Pinecone
# - pydantic-settings: Essencial para a configuração do Pinecone com Pydantic v2
!pip install -q --upgrade faiss-cpu chromadb langchain-pinecone pinecone-client "pydantic-settings>2.0.0"

import os
import shutil
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

# --- Componentes Comuns ---
# Vamos definir o modelo de embedding e os documentos que usaremos em todos os exemplos.

logger.info("Inicializando o modelo de embedding do Google...")
# CORREÇÃO IMPORTANTE: A dimensão do modelo 'models/embedding-001' é 768.
# Modelos mais antigos como 'gemini-embedding-001' também eram 768.
# A dimensão 3072 pertence a outros modelos, como os da OpenAI (text-embedding-ada-002 tem 1536).
# Usaremos 768 como a dimensão correta para o Gemini.
google_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
EMBEDDING_DIMENSION = 768
logger.info(f"✅ Modelo de embedding pronto. Dimensão do vetor: {EMBEDDING_DIMENSION}")

# Documentos de exemplo com metadados ricos
documentos_empresa = [
    Document(
        page_content="Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2024, "id_doc": "doc001"}
    ),
    Document(
        page_content="Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.",
        metadata={"tipo": "processo", "departamento": "Financeiro", "ano": 2023, "id_doc": "doc002"}
    ),
    Document(
        page_content="Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.",
        metadata={"tipo": "tutorial", "departamento": "TI", "ano": 2024, "id_doc": "doc003"}
    ),
    Document(
        page_content="Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2022, "id_doc": "doc004"}
    )
]

12:53:52 | INFO    | Inicializando o modelo de embedding do Google...
12:53:52 | INFO    | ✅ Modelo de embedding pronto. Dimensão do vetor: 768


###Vector Store 1 - FAISS (Rápido e em Memória)

In [8]:
from langchain_community.vectorstores import FAISS

logger.info("--- Testando Vector Store: FAISS ---")

# A LangChain abstrai a complexidade de criar o índice FAISS.
# `FAISS.from_documents` automaticamente:
# 1. Vetoriza cada documento usando o `google_embeddings`.
# 2. Cria um índice FAISS otimizado para busca.
# 3. Armazena os vetores e os documentos no índice.
faiss_db = FAISS.from_documents(documentos_empresa, google_embeddings)

pergunta = "Como peço minhas férias?"
resultados = faiss_db.similarity_search(pergunta, k=2)

print("\n" + "="*50)
print(f"🔍 Pergunta: '{pergunta}'")
print("📄 Documentos mais relevantes (FAISS):")
for doc in resultados:
    print(f"- Conteúdo: {doc.page_content}")
    print(f"  (Metadados: {doc.metadata})")
print("="*50)

12:53:52 | INFO    | --- Testando Vector Store: FAISS ---
12:53:52 | INFO    | Loading faiss with AVX2 support.
12:53:52 | INFO    | Successfully loaded faiss with AVX2 support.



🔍 Pergunta: 'Como peço minhas férias?'
📄 Documentos mais relevantes (FAISS):
- Conteúdo: Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
  (Metadados: {'tipo': 'política', 'departamento': 'RH', 'ano': 2024, 'id_doc': 'doc001'})
- Conteúdo: Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.
  (Metadados: {'tipo': 'processo', 'departamento': 'Financeiro', 'ano': 2023, 'id_doc': 'doc002'})


###Vector Store 2 - ChromaDB (Persistente e com Filtros)

In [9]:
from langchain_community.vectorstores import Chroma

logger.info("\n--- Testando Vector Store: ChromaDB ---")

# Boa prática: Limpar o diretório de persistência antes de rodar para garantir um estado limpo.
persist_directory = "./chroma_db_persist"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
logger.info(f"Diretório '{persist_directory}' limpo.")

# Criando o ChromaDB com persistência
chroma_db = Chroma.from_documents(
    documents=documentos_empresa,
    embedding=google_embeddings,
    persist_directory=persist_directory,
)
logger.info(f"✅ ChromaDB criado e persistido. Contém {chroma_db._collection.count()} documentos.")

# Teste 1: Busca por similaridade simples
pergunta = "Como peço minhas férias?"
resultados = chroma_db.similarity_search(pergunta, k=2)
print("\n" + "="*50)
print(f"🔍 Pergunta: '{pergunta}'")
print("📄 Documentos mais relevantes (ChromaDB - Busca Simples):")
for doc in resultados:
  print(f"- {doc.page_content}")
print("="*50)

# Teste 2: Busca com filtro de metadados (SINTAXE CORRIGIDA)
pergunta_rh = "Quais são as regras da empresa?"

# A sintaxe correta usa o operador "$and" com uma lista de condições.
filtro = {
    "$and": [
        {"departamento": {"$eq": "RH"}},
        {"tipo": {"$eq": "política"}}
    ]
}

resultados_filtrados = chroma_db.similarity_search(
    pergunta_rh,
    k=2,
    filter=filtro
)
print("\n" + "="*50)
print(f"🔍 Pergunta: '{pergunta_rh}' com filtro para departamento='RH' E tipo='política'")
print("📄 Documentos relevantes (ChromaDB - Busca com Filtro):")
for doc in resultados_filtrados:
  print(f"- Conteúdo: {doc.page_content}")
  print(f"  (Departamento: {doc.metadata['departamento']}, Tipo: {doc.metadata['tipo']})")
print("="*50)

12:53:52 | INFO    | 
--- Testando Vector Store: ChromaDB ---
12:53:52 | INFO    | Diretório './chroma_db_persist' limpo.
12:53:57 | INFO    | ✅ ChromaDB criado e persistido. Contém 4 documentos.



🔍 Pergunta: 'Como peço minhas férias?'
📄 Documentos mais relevantes (ChromaDB - Busca Simples):
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
- Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.

🔍 Pergunta: 'Quais são as regras da empresa?' com filtro para departamento='RH' E tipo='política'
📄 Documentos relevantes (ChromaDB - Busca com Filtro):
- Conteúdo: Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
  (Departamento: RH, Tipo: política)
- Conteúdo: Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
  (Departamento: RH, Tipo: política)


###Vector Store 3 - Pinecone (Nuvem e Escalável)

Pinecone é uma solução de nível profissional. Para usá-lo, precisamos configurar nossas chaves de API de forma segura. A abordagem a seguir, usando `userdata` do Colab e `Pydantic`, é a **melhor prática** para gerenciar configurações.

###1: Configuração Segura com Pydantic v2

In [10]:
from google.colab import userdata
from pydantic import Field, ValidationError, field_validator
from pydantic_settings import BaseSettings

# Carrega a chave da API do Pinecone a partir dos Secrets do Colab.
try:
    os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
    logger.info("PINECONE_API_KEY carregada com sucesso.")
except Exception as e:
    logger.error("Chave PINECONE_API_KEY não encontrada nos Secrets do Colab!")
    raise e

# Ótimo uso do Pydantic v2! A classe `BaseSettings` de `pydantic-settings`
# é a forma recomendada para carregar configurações de variáveis de ambiente.
# Ela valida os dados, garante os tipos corretos e previne erros.
class PineconeSettings(BaseSettings):
    pinecone_api_key: str = Field(..., env="PINECONE_API_KEY")
    index_name: str = Field("langchain-rag-aula", env="PINECONE_INDEX_NAME") # Nome do índice
    cloud: str = Field("aws", env="PINECONE_CLOUD")
    region: str = Field("us-east-1", env="PINECONE_REGION")

    @field_validator("pinecone_api_key")
    @classmethod
    def check_key_not_empty(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("PINECONE_API_KEY não pode ser vazio")
        return v

try:
    settings = PineconeSettings()
    logger.info("✅ Configurações do Pinecone validadas com sucesso.")
except ValidationError as e:
    logger.error(f"❌ Erro nas configurações do Pinecone:\n {e}")
    raise

12:53:58 | INFO    | PINECONE_API_KEY carregada com sucesso.
12:53:58 | INFO    | ✅ Configurações do Pinecone validadas com sucesso.


###2: Lógica de Criação/Conexão do Índice Pinecone

In [11]:
import time
from pinecone import Pinecone as PineconeClient, ServerlessSpec
from langchain_pinecone import Pinecone
from langchain_core.embeddings import Embeddings

# --- 1. Função Auxiliar para Gerenciar o Índice Pinecone ---
def get_or_create_pinecone_db(
    pinecone_client: PineconeClient,
    index_name: str,
    embedding_model: Embeddings,
    documents: list,
    spec: ServerlessSpec
) -> Pinecone:
    """
    Verifica um índice Pinecone, cria ou recria se necessário,
    e retorna um objeto LangChain Pinecone.
    """
    logger.info("--- Gerenciando Índice Pinecone ---")

    try:
        test_vector = embedding_model.embed_query("texto de teste")
        dynamic_dimension = len(test_vector)
        logger.info(f"Dimensão do vetor detectada dinamicamente: {dynamic_dimension}")
    except Exception as e:
        logger.error(f"Não foi possível determinar a dimensão do embedding: {e}")
        raise

    if index_name in pinecone_client.list_indexes().names():
        logger.info(f"Índice '{index_name}' encontrado. Verificando metadados...")
        index_info = pinecone_client.describe_index(index_name)

        if index_info.dimension == dynamic_dimension:
            logger.info("Dimensão compatível. Conectando ao índice existente...")
            return Pinecone.from_existing_index(index_name, embedding_model)
        else:
            logger.warning(f"Incompatibilidade de dimensão! Índice tem {index_info.dimension}, mas o modelo gera {dynamic_dimension}.")
            logger.warning(f"Excluindo o índice '{index_name}' para recriá-lo...")
            pinecone_client.delete_index(index_name)
            while index_name in pinecone_client.list_indexes().names():
                time.sleep(1)
            logger.info("Índice antigo excluído.")

    logger.info(f"Criando novo índice '{index_name}' com dimensão {dynamic_dimension}...")
    pinecone_client.create_index(
        name=index_name, dimension=dynamic_dimension, metric="cosine", spec=spec
    )
    while not pinecone_client.describe_index(index_name).status['ready']:
        time.sleep(1)
    logger.info(f"Índice '{index_name}' criado. Populando com documentos...")

    pinecone_db = Pinecone.from_documents(
        documents, embedding_model, index_name=index_name
    )
    logger.info("✅ Documentos inseridos com sucesso.")
    return pinecone_db

# --- 2. Execução Principal ---
pinecone_client = PineconeClient(api_key=settings.pinecone_api_key)
spec = ServerlessSpec(cloud=settings.cloud, region=settings.region)

pinecone_db = get_or_create_pinecone_db(
    pinecone_client=pinecone_client,
    index_name=settings.index_name,
    embedding_model=google_embeddings,
    documents=documentos_empresa,
    spec=spec
)

# --- 3.1 Uso do Vector Store ---
if pinecone_db:
    # --- 3.1 Busca por Similaridade Simples ---
    pergunta_ti = "Como configuro a VPN?"
    resultados_pinecone = pinecone_db.similarity_search(pergunta_ti, k=1)

    print("\n" + "="*50)
    print(f"🔍 Pergunta: '{pergunta_ti}'")
    print("📄 Documentos mais relevantes (Pinecone - Busca Simples):")
    for doc in resultados_pinecone:
        print(f"- Conteúdo: {doc.page_content}")
        print(f"  (Metadados: {doc.metadata})")
    print("="*50)

    print("\n" + "-"*20)

    # --- 3.2 Busca com Filtro  ---
    # A sintaxe de filtro do Pinecone para uma igualdade simples é um dicionário direto.
    # É mais simples que a do ChromaDB para este caso de uso.
    resultados_pinecone_filtrados = pinecone_db.similarity_search(
        "informações sobre regras",
        k=2,
        filter={"tipo": "política"}
    )
    print(f"\n🔍 Pergunta: 'informações sobre regras' com filtro para tipo='política'")
    print("\n📄 Documentos relevantes (Pinecone - Busca com Filtro):")
    for doc in resultados_pinecone_filtrados:
        print(f"- Conteúdo: {doc.page_content}")
        print(f"  (Tipo: {doc.metadata['tipo']})")
    print("="*50)

else:
    logger.error("Não foi possível obter uma instância do Vector Store do Pinecone.")

12:53:58 | INFO    | --- Gerenciando Índice Pinecone ---
12:53:58 | INFO    | Dimensão do vetor detectada dinamicamente: 3072
12:54:02 | INFO    | Índice 'langchain-rag-aula' encontrado. Verificando metadados...
12:54:03 | INFO    | Dimensão compatível. Conectando ao índice existente...



🔍 Pergunta: 'Como configuro a VPN?'
📄 Documentos mais relevantes (Pinecone - Busca Simples):
- Conteúdo: Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.
  (Metadados: {'ano': 2024.0, 'departamento': 'TI', 'id_doc': 'doc003', 'tipo': 'tutorial'})

--------------------

🔍 Pergunta: 'informações sobre regras' com filtro para tipo='política'

📄 Documentos relevantes (Pinecone - Busca com Filtro):
- Conteúdo: Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
  (Tipo: política)
- Conteúdo: Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
  (Tipo: política)


---

## Para saber mais **HNSW (Hierarchical Navigable Small World)**

## **1. Estrutura do HNSW**

O HNSW é um índice baseado em **grafos** que organiza vetores de forma hierárquica.

* Cada nó do grafo representa um item do conjunto de dados.
* Cada nó mantém ligações com seus vizinhos mais próximos.

Essa estrutura possibilita **saltos estratégicos** durante a busca, acelerando a recuperação dos itens mais semelhantes — ainda que não se atinja a mesma precisão de uma busca exaustiva.

---

## **2. Papel do Número de Vizinhos**

Um dos parâmetros essenciais na configuração do HNSW é o número de vizinhos conectados a cada nó (*geralmente definido como 32*). Esse parâmetro afeta diretamente:

* **Qualidade da busca**

  * Um número maior de vizinhos tende a aumentar o *recall* (probabilidade de encontrar itens realmente próximos ao vetor de consulta).
  * Especialmente útil em dados com **alta variabilidade**.

* **Desempenho e custo computacional**

  * Mais conexões aumentam o uso de memória e o tempo de construção do índice.
  * Valores muito altos podem gerar lentidão em ambientes com **recursos limitados**.

---

## **3. Considerações na Escolha do Valor**

A definição do número de vizinhos é um **trade-off entre velocidade e acurácia**:

* Projetos que priorizam **precisão** e dispõem de **recursos robustos** → usar valores mais altos.
* Cenários com **grandes volumes de dados** ou **protótipos iniciais** → valores menores oferecem respostas mais rápidas, embora com leve perda de precisão.

A escolha ideal exige **testes práticos** e aferição de métricas de similaridade, sempre alinhada aos objetivos do projeto.

---

## **4. Exemplo Prático em Código**

```python
import faiss

# Dimensão dos vetores
d = 768

# Número de vizinhos configurados para o índice HNSW
M = 32

# Criação do índice HNSW
index = faiss.IndexHNSWFlat(d, M)
```

Nesse exemplo:

* Vetores de dimensão **768**.
* Cada nó mantém **32 conexões**.
* Essa configuração serve como **ponto de partida** e pode ser ajustada conforme a avaliação de desempenho e a natureza dos dados.

---

## **5. Conclusão**

A experimentação com diferentes valores de vizinhos é crucial para encontrar o **equilíbrio ideal**:

* **Buscas rápidas**.
* **Resultados relevantes**.
* **Uso eficiente de recursos**.

---



In [12]:
import os
from google.colab import userdata

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

# Seção 3: Embeddings de Alta Performance - Comparando Velocidade e Qualidade

A escolha do modelo de embedding impacta diretamente dois eixos críticos de um sistema RAG:

1.  **Performance (Velocidade):** O quão rápido conseguimos transformar um grande volume de documentos em vetores? Isso é crucial para a etapa de ingestão e indexação de dados.
2.  **Qualidade (Semântica):** O quão bem o modelo entende a nuance e a intenção por trás de uma pergunta para encontrar os documentos mais relevantes?

Nesta seção, faremos um benchmark completo, comparando quatro modelos populares em ambos os eixos, com um foco especial em otimização de hardware (CPU vs. GPU).

##Preparação do Ambiente e do Benchmark

In [13]:
# --- 1. Instalações ---
!pip install -q --upgrade langchain langchain-google-genai sentence-transformers scikit-learn pandas plotly umap-learn
# --- 2. Imports e Verificação de Hardware ---
import time
import torch
import pandas as pd
import plotly.express as px
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Verificação de GPU: Essencial para garantir que a otimização funcione.
if torch.cuda.is_available():
    DEVICE = "cuda"
    logger.info(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}. Usando {DEVICE}.")
else:
    DEVICE = "cpu"
    logger.warning("⚠️ GPU não detectada. Modelos locais rodarão na CPU (mais lento).")

# --- 3. Dados de Teste ---
textos_teste = [
    "Qual é a política de férias da nossa empresa?",
    "Preciso de um relatório de despesas de viagem.",
    "Como configuro o acesso à rede privada virtual (VPN)?",
    "Onde encontro o código de conduta da organização?",
    "Quero entender o processo de avaliação de performance."
]

12:54:18 | WARNING | ⚠️ GPU não detectada. Modelos locais rodarão na CPU (mais lento).


##Benchmark de Performance

In [14]:
# --- 1. Definição dos Modelos para o Benchmark ---

# Otimizamos os modelos Hugging Face para usar a GPU se disponível.
modelos_para_testar = {
    "Gemini (API)": GoogleGenerativeAIEmbeddings(model="gemini-embedding-001"),
    "Multilingual-E5 (GPU)": HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-large",
        model_kwargs={'device': DEVICE}
    ),
    "MiniLM (GPU)": HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': DEVICE}
    ),
    "BGE-Large (GPU)": HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5",
        model_kwargs={'device': DEVICE},
        encode_kwargs={'normalize_embeddings': True}
    )
}

# --- 2. Execução do Benchmark ---
logger.info(f"Iniciando benchmark com {len(textos_teste)} documentos...")
resultados_performance = []
embeddings_gerados = {} # Dicionário para armazenar os vetores para a próxima fase

for nome, modelo in modelos_para_testar.items():
    logger.info(f"Processando com o modelo: {nome}...")
    start_time = time.time()

    # Gera os embeddings
    vetores = modelo.embed_documents(textos_teste)

    end_time = time.time()
    tempo_total = end_time - start_time

    # Armazena os vetores para uso posterior na análise de qualidade
    embeddings_gerados[nome] = vetores

    # Coleta as métricas de performance
    resultados_performance.append({
        "Modelo": nome,
        "Tempo Total (s)": tempo_total,
        "Dimensão do Vetor": len(vetores[0]) if vetores else 0,
        "Documentos": len(textos_teste)
    })

logger.info("✅ Benchmark de performance concluído.")

# --- 3. Exibição dos Resultados ---
df_performance = pd.DataFrame(resultados_performance)

print("\n" + "="*60)
print("Resultados do Benchmark de Performance")
print("="*60)
print(df_performance.to_string(index=False))

# --- 4. Visualização Gráfica ---
fig = px.scatter(
    df_performance,
    x="Dimensão do Vetor",
    y="Tempo Total (s)",
    text="Modelo",
    size_max=60,
    hover_name="Modelo",
    title="Performance de Embedding: Tempo vs. Dimensão do Vetor"
)
fig.update_traces(textposition='top center')
fig.show()

/tmp/ipython-input-3271807629.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  "Multilingual-E5 (GPU)": HuggingFaceEmbeddings(
12:54:18 | INFO    | Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to


Resultados do Benchmark de Performance
               Modelo  Tempo Total (s)  Dimensão do Vetor  Documentos
         Gemini (API)         0.349147               3072           5
Multilingual-E5 (GPU)         3.809625               1024           5
         MiniLM (GPU)         0.305746                384           5
      BGE-Large (GPU)         6.954527               1024           5


### Seção 3.2: Análise Visual e Quantitativa Avançada

Além da velocidade, é crucial entender se os diferentes modelos "organizam" o conhecimento da mesma forma. Ou seja, se os vetores que eles criam para os mesmos textos mantêm relações de similaridade parecidas.

O script a seguir é uma pipeline de análise avançada que faz o seguinte:
1.  **Geração e Cache de Embeddings:** Garante que os vetores sejam calculados apenas uma vez e salvos em disco para reuso rápido.
2.  **Redução de Dimensionalidade (PCA/UMAP):** Projeta os vetores de alta dimensão (ex: 1024D) em um espaço 2D ou 3D que podemos visualizar.
3.  **Alinhamento (Procrustes):** Tenta "rotacionar" os espaços vetoriais de cada modelo para que fiquem o mais alinhados possível, permitindo uma comparação visual justa, especialmente quando as dimensões originais dos vetores são diferentes.
4.  **Cálculo de Métricas e Visualização:** Gera métricas quantitativas (distâncias, scores) e plots interativos para nos ajudar a responder: **"Quão parecida é a 'visão de mundo' de cada modelo de embedding?"**

###Setup da Análise e Geração de Embeddings com Cache

In [17]:
# Célula 17 (DEFINITIVAMENTE CORRIGIDA) - Setup da Análise e Geração de Embeddings com Cache

# --- 1. Instalações para Análise Avançada ---
!pip install -q --upgrade umap-learn "tqdm>=4.62.3"

# --- 2. Imports e Configurações ---
import os
import time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
import umap
import plotly.express as px
import plotly.graph_objects as go
from scipy.linalg import orthogonal_procrustes

# --- 3. Constantes e Diretórios ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CACHE_DIR = "embeddings_cache"
METRICS_DIR = "metrics_output"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# --- 4. Função de Cache (SIMPLIFICADA E CORRIGIDA) ---
def get_embeddings_for_model(model_name, emb_client, texts):
    """
    Gera ou carrega embeddings do cache. A chamada para .embed_documents() é a mesma para todos os modelos.
    """
    safe_model_name = model_name.replace('/', '_').replace(' ', '_')
    cache_path = os.path.join(CACHE_DIR, f"{safe_model_name}.npy")

    if os.path.exists(cache_path):
        logger.info(f"Carregando cache para '{model_name}' de {cache_path}")
        return np.load(cache_path)

    logger.info(f"Cache não encontrado. Gerando embeddings para '{model_name}'...")

    # A CORREÇÃO ESTÁ AQUI:
    # A chamada para .embed_documents() não usa 'chunk_size' para nenhum dos modelos.
    # A biblioteca subjacente (sentence-transformers) otimiza o batching internamente.
    vetores = emb_client.embed_documents(texts)

    arr = np.array(vetores)
    np.save(cache_path, arr)
    logger.info(f"Salvo cache em {cache_path} com shape={arr.shape}")
    return arr

# --- 5. Geração ou Carregamento dos Vetores ---
models_arrays = []
model_names = []

logger.info("Iniciando extração de embeddings para análise avançada...")
# Usamos as variáveis `modelos_para_testar` e `textos_teste` definidas anteriormente
for name, client in tqdm(modelos_para_testar.items(), desc="Processando Modelos"):
    arr = get_embeddings_for_model(name, client, textos_teste)
    models_arrays.append(arr)
    model_names.append(name)

logger.info("✅ Todos os embeddings foram carregados/gerados.")

13:00:57 | INFO    | Iniciando extração de embeddings para análise avançada...


Processando Modelos:   0%|          | 0/4 [00:00<?, ?it/s]

13:00:57 | INFO    | Carregando cache para 'Gemini (API)' de embeddings_cache/Gemini_(API).npy
13:00:57 | INFO    | Cache não encontrado. Gerando embeddings para 'Multilingual-E5 (GPU)'...
13:00:58 | INFO    | Salvo cache em embeddings_cache/Multilingual-E5_(GPU).npy com shape=(5, 1024)
13:00:58 | INFO    | Cache não encontrado. Gerando embeddings para 'MiniLM (GPU)'...
13:00:59 | INFO    | Salvo cache em embeddings_cache/MiniLM_(GPU).npy com shape=(5, 384)
13:00:59 | INFO    | Cache não encontrado. Gerando embeddings para 'BGE-Large (GPU)'...
13:01:00 | INFO    | Salvo cache em embeddings_cache/BGE-Large_(GPU).npy com shape=(5, 1024)
13:01:00 | INFO    | ✅ Todos os embeddings foram carregados/gerados.


###Redução de Dimensionalidade e Alinhamento Procrustes

In [18]:
# --- 1. Verificação de Dimensões ---
dims = [arr.shape[1] for arr in models_arrays]
same_dim = all(d == dims[0] for d in dims)
n_models = len(models_arrays)
n_docs = len(textos_teste)
logger.info(f"Dimensões por modelo: {dict(zip(model_names, dims))}")

# --- 2. Geração de Componentes 3D e Alinhamento ---
aligned_list = []
if same_dim:
    logger.info("Todas as dimensões são iguais -> Usando PCA conjunta para alinhamento.")
    X_all = np.vstack(models_arrays)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_all)
    pca = PCA(n_components=3, random_state=RANDOM_STATE)
    X_pca3 = pca.fit_transform(X_scaled)

    for i in range(n_models):
        aligned_list.append(X_pca3[i * n_docs : (i + 1) * n_docs, :])
else:
    logger.info("Dimensões diferentes -> Usando PCA por modelo + Alinhamento Procrustes.")
    per_model_comps = []
    for arr, name in zip(models_arrays, model_names):
        scaler = StandardScaler()
        arr_s = scaler.fit_transform(arr)
        pca = PCA(n_components=3, random_state=RANDOM_STATE)
        per_model_comps.append(pca.fit_transform(arr_s))

    ref = per_model_comps[0]
    aligned_list.append(ref)
    for i in range(1, len(per_model_comps)):
        target = per_model_comps[i]
        R, scale = orthogonal_procrustes(target, ref)
        aligned_list.append((target @ R) * scale)

aligned_3d = np.vstack(aligned_list)
np.save(os.path.join(METRICS_DIR, "aligned_components_3d.npy"), aligned_3d)
logger.info(f"✅ Componentes 3D alinhados e salvos. Shape final: {aligned_3d.shape}")

# --- 3. Preparação do DataFrame para Plotagem ---
labels = [name for name in model_names for _ in range(n_docs)]
docs = textos_teste * n_models

df_plot = pd.DataFrame({
    "X": aligned_3d[:,0], "Y": aligned_3d[:,1], "Z": aligned_3d[:,2],
    "Modelo": labels, "Documento": docs
})
df_plot.to_csv(os.path.join(METRICS_DIR, "df_plot_aligned.csv"), index=False)
logger.info("✅ DataFrame para plotagem criado e salvo.")

13:01:40 | INFO    | Dimensões por modelo: {'Gemini (API)': 3072, 'Multilingual-E5 (GPU)': 1024, 'MiniLM (GPU)': 384, 'BGE-Large (GPU)': 1024}
13:01:40 | INFO    | Dimensões diferentes -> Usando PCA por modelo + Alinhamento Procrustes.
13:01:41 | INFO    | ✅ Componentes 3D alinhados e salvos. Shape final: (20, 3)
13:01:41 | INFO    | ✅ DataFrame para plotagem criado e salvo.


###Métricas Quantitativas e Visualizações

In [19]:
# --- 1. Cálculo de Distâncias Intra e Inter-Modelo ---
D_cos = pairwise_distances(aligned_3d, metric='cosine')
n_points = D_cos.shape[0]
intra, inter = [], []
models_arr = np.array(df_plot["Modelo"].tolist())
for i in range(n_points):
    for j in range(i + 1, n_points):
        if models_arr[i] == models_arr[j]:
            intra.append(D_cos[i, j])
        else:
            inter.append(D_cos[i, j])

# --- 2. Plot: Histograma de Distâncias ---
hist = go.Figure()
hist.add_trace(go.Histogram(x=intra, name='Intra-modelo (mesmo modelo, docs diferentes)', opacity=0.75))
hist.add_trace(go.Histogram(x=inter, name='Inter-modelo (mesmo doc, modelos diferentes)', opacity=0.75))
hist.update_layout(barmode='overlay', title='Distribuição das Distâncias de Cosseno')
hist.show()

# --- 3. Plot: Scatter 3D Interativo ---
fig3d = px.scatter_3d(
    df_plot, x="X", y="Y", z="Z", color="Modelo",
    hover_data=["Documento"],
    title="Embeddings 3D (Componentes Alinhados)"
)
fig3d.update_traces(marker=dict(size=5))
fig3d.show()

# --- 4. Plot: UMAP 2D (Visão Alternativa) ---
logger.info("Calculando projeção UMAP 2D...")
reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
X_umap2 = reducer.fit_transform(aligned_3d)
df_plot["UMAP1"], df_plot["UMAP2"] = X_umap2[:,0], X_umap2[:,1]

fig_umap = px.scatter(
    df_plot, x="UMAP1", y="UMAP2", color="Modelo",
    hover_data=["Documento"], title="Projeção UMAP 2D dos Espaços Vetoriais Alinhados"
)
fig_umap.show()

13:02:08 | INFO    | Calculando projeção UMAP 2D...
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



##Seção 3.3 - Aprofundando a Análise: Como os Modelos se Comparam Entre Si?

Já vimos a performance e a estrutura visual dos embeddings. Agora, vamos quantificar a "similaridade" entre as visões de mundo de cada modelo.

Começaremos com uma visualização poderosa: um **heatmap da matriz de distância entre os centroides**. O centroide é o "ponto médio" de todos os vetores de um modelo. A distância entre os centroides nos dá uma ideia de quão distantes, em média, são os espaços semânticos de cada modelo.

**O que procurar no heatmap:**
-   **Valores baixos (cores escuras):** Indicam que os dois modelos organizam o significado dos textos de forma muito parecida.
-   **Valores altos (cores claras):** Indicam que as "visões de mundo" dos modelos são semanticamente mais distantes.

###Visualização - Heatmap da Distância entre Centroides

In [20]:
import pandas as pd
import plotly.express as px
import os

# Carrega o arquivo CSV com a matriz de distância, gerado na célula de análise anterior.
metrics_dir = "metrics_output"
centroid_matrix_path = os.path.join(metrics_dir, "centroid_distance_matrix.csv")

if os.path.exists(centroid_matrix_path):
    logger.info(f"Carregando matriz de distância de '{centroid_matrix_path}'...")
    centroid_df = pd.read_csv(centroid_matrix_path, index_col=0)

    # Cria o heatmap interativo com Plotly
    fig = px.imshow(
        centroid_df,
        text_auto=True, # Mostra os valores de distância diretamente no gráfico
        color_continuous_scale="Viridis_r", # Usamos '_r' para inverter a escala (menor = mais escuro)
        title="Heatmap da Distância de Cosseno entre os Centroides dos Modelos"
    )
    fig.update_layout(xaxis_title="Modelo A", yaxis_title="Modelo B")
    fig.show()
    logger.info("✅ Heatmap gerado com sucesso.")
else:
    logger.warning(f"Arquivo '{centroid_matrix_path}' não encontrado. Execute a célula de análise avançada (Célula 18) primeiro.")

13:27:22 | WARNING | Arquivo 'metrics_output/centroid_distance_matrix.csv' não encontrado. Execute a célula de análise avançada (Célula 18) primeiro.


---

### 📊 Observações sobre o teste de embeddings com múltiplas queries

O teste utilizou várias queries de exemplo, como consultas sobre folga, VPN, relatórios e código de conduta, avaliando quatro modelos de embeddings: **Gemini**, **MiniLM (all-MiniLM-L6-v2)**, **Multilingual-e5-large** e **BGE-large**.

---

### Principais resultados qualitativos

* **Gemini (embedding-001)**
  * Em queries curtas e focadas, às vezes prioriza a estrutura da frase ao invés do conceito central.
  * Mostra boa capacidade de captura semântica, mas scores podem variar dependendo do tema.

* **MiniLM (all-MiniLM-L6-v2)**
  * Rápido e leve, adequado para cenários de baixa latência.
  * Perde precisão semântica em queries mais específicas ou em idiomas fora do inglês predominante do treinamento.

* **Multilingual-e5-large**
  * Robusto para múltiplos idiomas, inclusive português.
  * Captura bem relações semânticas e traz respostas relevantes no Top-1 com maior consistência de score.

* **BGE-large (BAAI/bge-large-en-v1.5)**
  * Embeddings ajustados para retrieval, conseguem priorizar corretamente o conceito central da query.
  * Mesmo que scores absolutos possam ser menores que outros modelos, a relevância do Top-1 geralmente é alta.

---

### Por que isso acontece?

* **Treinamento e foco linguístico**
  * BGE-large foi otimizado para retrieval e captura nuances semânticas de forma consistente.
  * Gemini ainda está em pré-lançamento, podendo favorecer padrões de frase mais do que o conceito.
  * MiniLM é projetado para velocidade, com embeddings mais genéricos.
  * Multilingual-e5-large é sólido para consultas multilíngues, incluindo português.

* **Capacidade e dimensão do modelo**
  * Modelos maiores, como BGE-large (≈1B parâmetros) e Gemini (3072 dimensões), carregam mais nuances semânticas.
  * Modelos menores ou mais leves, como MiniLM (33M parâmetros, 384 dimensões), são rápidos, mas menos precisos.

* **Dados de treinamento**
  * Nem todos os modelos foram igualmente expostos a português ou a domínios corporativos.
  * BGE-large e Multilingual-e5-large tendem a lidar melhor com queries multilíngues.

---

### ⚠️ Considerações

* Um único teste não permite conclusões estatísticas definitivas.
* Para análise robusta, recomenda-se rodar **várias queries reais**, computando métricas como **Top-k accuracy** e **Mean Reciprocal Rank (MRR)**.
* Scores absolutos não são comparáveis entre modelos; a análise deve focar na **ordem relativa do ranking** para cada modelo.
* Em cenários críticos, considere **rerankers** ou validação humana para garantir a relevância das respostas.

---

### ✅ Recomendações práticas

1. Continue testando com queries representativas do seu domínio.
2. Avalie **Top-k** e **MRR** para cada modelo.
3. Considere trade-offs:
   * **BGE-large**: ótima precisão e relevância, ideal offline.
   * **Multilingual-e5-large**: alta consistência em múltiplos idiomas.
   * **Gemini**: modelo em evolução, potencial de melhoria rápida.
   * **MiniLM**: velocidade e baixo custo, mas menor precisão semântica.


---

### 📊 Desempenho dos modelos de embeddings

#### **BGE-large**
- **Top-1**: ✅
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 1.0
- **Observações**: Captura bem o conceito central; embeddings ajustados para retrieval.

---

#### **Gemini**
- **Top-1**: ✅
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 1.0
- **Observações**: Às vezes prioriza a estrutura da frase, mas já mostra boa semântica.

---

#### **Multilingual-e5-large**
- **Top-1**: ✅
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 1.0
- **Observações**: Robustez em múltiplos idiomas, inclusive português.

---

#### **MiniLM**
- **Top-1**: ❌
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 0.17
- **Observações**: Rápido e leve, mas perde precisão em queries curtas ou específicas.



---

### 🔍 Explicando as métricas

* **Top-1**: Indica se o documento mais relevante foi retornado na primeira posição.  
  ✅ = correto, ❌ = incorreto  

* **Top-3**: Mostra se o documento relevante está entre os três primeiros retornos.  

* **Top-5**: Verifica se o documento relevante aparece entre os cinco primeiros.  

* **MRR (Mean Reciprocal Rank)**: Métrica que avalia a posição do documento relevante; quanto mais próximo do topo, maior o valor (máximo = 1.0).

---

### ⚖️ Considerações pedagógicas

* **Treinamento e foco linguístico**: modelos grandes ou multilíngues capturam melhor nuances semânticas.  
* **Dimensão do vetor**: embeddings maiores tendem a guardar mais informação semântica, mas exigem mais memória.  
* **Dados de treino**: exposição a português ou termos corporativos afeta diretamente o ranking.  

---


## Interpretação dos resultados do benchmark de embeddings

O gráfico **grouped bar** mostra a comparação de **Top@1, Top@3, Top@5 e MRR** entre os modelos testados: BGE-large, Gemini, MiniLM e Multilingual-e5-large.

### 1️⃣ Modelos de melhor desempenho
- **BGE-large, Gemini e Multilingual-e5-large**
  - **Top@1 = 1.0** → o documento mais relevante foi sempre o primeiro na lista.
  - **Top@3 e Top@5 = 1.0** → o documento relevante aparece consistentemente entre os primeiros.
  - **MRR = 1.0** → reforça que os modelos colocam o relevante no topo.
  - **Conclusão:** esses modelos capturam bem a semântica das queries, mesmo curtas ou em português.

### 2️⃣ Modelo com desempenho inferior
- **MiniLM**
  - **Top@1 = 0.0** → na maioria das queries, o documento relevante **não foi o primeiro**.
  - **Top@3 e Top@5 = 0.5** → o relevante aparece dentro do ranking, mas nem sempre em posições altas.
  - **MRR = 0.1667** → penaliza o modelo pela baixa posição do relevante.
  - **Conclusão:** MiniLM é mais leve e rápido, mas perde precisão em captura semântica, especialmente em queries curtas ou multilíngues.

### 3️⃣ Observações gerais
- O **Top@3 e Top@5** podem saturar rapidamente em rankings curtos, tornando-os menos discriminativos.
- O **MRR** é mais sensível à posição do relevante, permitindo diferenciar melhor o MiniLM.
- Para análises mais robustas, recomenda-se:
  - Aumentar o número de queries;
  - Aumentar o número de documentos por ranking;
  - Avaliar métricas como **Top-k accuracy** e **MRR** em conjunto com rerankers ou filtros semânticos.

> **Resumo visual:** No gráfico, cada grupo de barras representa um modelo. As cores correspondem às métricas (Top@1, Top@3, Top@5, MRR). Modelos que atingem o topo em todas as queries terão todas as barras no valor 1, enquanto modelos com menor desempenho terão barras mais baixas, evidenciando suas limitações.


# Glossário Visual de Métricas de Recuperação de Informação

Quando avaliamos modelos de embeddings, usamos métricas que mostram **onde o documento relevante aparece no ranking**. Vamos visualizar cada uma delas.

---

## **Top@1** ✅
- **O que mede:** Se o documento relevante está **em 1º lugar**.
- **Visualização:**
```

1️⃣ Documento relevante ✔
2️⃣ Outro documento
3️⃣ Outro documento

```
- **Interpretação do valor:**  
  - 1 → sucesso (relevante em 1º)  
  - 0 → falha (relevante não está em 1º)

---

## **Top@3** 🟢
- **O que mede:** Se o documento relevante está **entre os 3 primeiros**.
- **Visualização:**
```

1️⃣ Outro documento
2️⃣ Documento relevante ✔
3️⃣ Outro documento

```
- **Interpretação do valor:**  
  - 1 → relevante aparece no Top 3  
  - 0 → não aparece no Top 3

---

## **Top@5** 🔵
- **O que mede:** Se o documento relevante está **entre os 5 primeiros**.
- **Visualização:**
```

1️⃣ Outro documento
2️⃣ Outro documento
3️⃣ Documento relevante ✔
4️⃣ Outro documento
5️⃣ Outro documento

```
- **Interpretação do valor:**  
  - 1 → relevante aparece no Top 5  
  - 0 → não aparece no Top 5

---

## **MRR (Mean Reciprocal Rank)** 💡
- **O que mede:** A **posição média do documento relevante**, penalizando posições mais baixas.
- **Cálculo rápido:**  
\[
\text{RR} = \frac{1}{\text{posição do relevante}}
\]  
  - 1º lugar → RR = 1.0  
  - 2º lugar → RR = 0.5  
  - 5º lugar → RR = 0.2
- **Para várias queries:** Média dos RR → **MRR**
- **Visualização simplificada:**
```

Ranking: [Doc A, Doc B ✔, Doc C, Doc D]
Posição do relevante: 2
RR = 1/2 = 0.5

```

---

## **Resumo rápido**
| Métrica | Faixa | O que mostra |
|---------|-------|--------------|
| Top@1   | 0–1   | Relevante em 1º? |
| Top@3   | 0–1   | Relevante entre os 3 primeiros? |
| Top@5   | 0–1   | Relevante entre os 5 primeiros? |
| MRR     | 0–1   | Posição média do relevante (quanto maior, melhor) |

> 💡 **Dica de estudo:**  
> - **Top@k** → simples e intuitivo; ótimo para análises rápidas.  
> - **MRR** → mais sensível à posição; ideal para rankings longos ou consultas críticas.

---

### ✅ Conclusão
- Use **Top@k** para avaliar se o modelo acerta o topo do ranking.  
- Use **MRR** para avaliar a posição exata do relevante dentro do ranking.  
- Juntas, essas métricas ajudam a entender **precisão e qualidade do ranking** de cada modelo.
